# Phase 2 · Step 1 — GGUF / llama.cpp (Sec. 6 item 1), KV-cache quantization (item 4), MXFP4/NVFP4 (item 2)

**What this notebook does**

| Variant | Weights | KV cache | Purpose |
|---|---|---|---|
| `gguf_f16` | F16 | f16 | llama.cpp *runtime* baseline (separates "runtime effect" from "quantization effect") |
| `gguf_q8_0` | Q8_0 | f16 | 8-bit k-quant vs bitsandbytes INT8 |
| `gguf_q4_k_m` | Q4_K_M | f16 | 4-bit k-quant vs bitsandbytes NF4 |
| `gguf_f16_kv8` | F16 | q8_0 | **KV-cache quantization retry** (item 4) — llama.cpp's own KV quant has no sliding-window restriction |
| `gguf_f16_kv4` | F16 | q4_0 | KV-cache 4-bit |
| `gguf_mxfp4` | MXFP4 (E2M1, 32-block, power-of-2 scale) | f16 | **NVFP4/MXFP4 item 2** — format accuracy without Blackwell |
| `gguf_nvfp4` | NVFP4 (E2M1, 16-block, FP8-E4M3 scale) | f16 | same |

**About the FP4 rows:** the pinned llama.cpp has native `MXFP4` and `NVFP4` tensor types with CUDA kernels that
also run on pre-Blackwell GPUs (software decode). So the *accuracy* numbers are real for these formats, but the
*speed* numbers are **not** what Blackwell FP4 tensor cores would give — on the T4 activations go through 8-bit
dot products, i.e. this is W4(FP4)/A8, not native W4A4. Say exactly this in the report.

**Controls kept identical to Phase 1:** same 200 GSM8K / 200 MMLU questions (seed 42), same prompts,
same chat template (checked in cell 6), same answer extractors, greedy decoding, `max_new_tokens=512`.

**Colab constraints handled**
* Every answer is appended to a JSONL file on Google Drive → if Colab disconnects or the GPU quota runs out,
  just re-run all cells; finished questions are skipped (resume).
* The compiled llama.cpp binaries are cached on Drive → the ~15-25 min CUDA build happens only once.
* **Just choose Runtime → Run all every session.** Cell 2 prints what is finished and runs only what is left, in priority order
  (FP16 → Q4_K_M → Q8_0 → KV q4 → KV q8 → NVFP4 → MXFP4). Each variant ≈ 30-60 min on a T4; expect 3-4 sessions.
* When cell 2 prints `ALL llama.cpp VARIANTS ARE DONE`, this notebook is finished.

Runtime → Change runtime type → **T4 GPU**, then run top to bottom.

In [ ]:
# ==== 1. Settings — the only cell you normally edit ====
SMOKE_TEST   = False         # False: full 200/task into runs/  (True: 10 questions/task into runs_smoke/)
RUN_VARIANTS = "auto"        # "auto" = every variant not finished yet, in priority order. Just Run all each session.
ORDER = ["gguf_f16", "gguf_q4_k_m", "gguf_q8_0", "gguf_f16_kv4", "gguf_f16_kv8", "gguf_nvfp4", "gguf_mxfp4",
         # joint weight x KV-cache grid (completes the 3x3 grid {F16, Q8_0, Q4_K_M} x {f16, q8_0, q4_0})
         "gguf_q4_k_m_kv8", "gguf_q4_k_m_kv4", "gguf_q8_0_kv8", "gguf_q8_0_kv4"]

MODEL_ID     = "microsoft/Phi-4-mini-instruct"
LLAMA_COMMIT = "d327bf1f0780f87b8c01b367b61853b0c291d7c4"   # pinned llama.cpp commit (report this in the dissertation)
CTX_SIZE     = 4096           # prompt + 512 generated tokens fits easily
MAX_NEW_TOKENS = 512          # same as Phase 1
PORT         = 8181           # NOT 8080: Colab itself uses port 8080

VARIANTS = {
    "gguf_f16":     dict(weights="F16",    kv="f16"),
    "gguf_q8_0":    dict(weights="Q8_0",   kv="f16"),
    "gguf_q4_k_m":  dict(weights="Q4_K_M", kv="f16"),
    "gguf_f16_kv8": dict(weights="F16",    kv="q8_0"),
    "gguf_f16_kv4": dict(weights="F16",    kv="q4_0"),
    # Sec. 6 item 2 — FP4 microscaling formats, run with llama.cpp's non-Blackwell kernels (see notes at top)
    "gguf_mxfp4":   dict(weights="MXFP4",  kv="f16"),
    "gguf_nvfp4":   dict(weights="NVFP4",  kv="f16"),
    # joint weight + KV-cache quantization grid
    "gguf_q4_k_m_kv8": dict(weights="Q4_K_M", kv="q8_0"),
    "gguf_q4_k_m_kv4": dict(weights="Q4_K_M", kv="q4_0"),
    "gguf_q8_0_kv8":   dict(weights="Q8_0",   kv="q8_0"),
    "gguf_q8_0_kv4":   dict(weights="Q8_0",   kv="q4_0"),
}

In [ ]:
# ==== 2. Paths + Google Drive ====
import os, sys, json, time, re, subprocess, threading, gc, shutil, random

USE_DRIVE = True    # set False if mounting Drive keeps failing ("Failed to fetch")

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DRIVE_OK = False
if IN_COLAB:
    BASE = "/content"
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            DRIVE_OK = True
        except Exception as e:      # browser blocked the Drive sign-in popup
            print("!! Google Drive could not be mounted:", e)
    DRIVE = "/content/drive/MyDrive/quant_study_phase2" if DRIVE_OK else "/content/no_drive"
    if not DRIVE_OK:
        print("!! Working WITHOUT Drive: results live only in this session.\n"
              "!! Run the LAST cell before closing Colab to download them.")
else:                               # running outside Colab
    BASE  = os.environ.get("QS_BASE", os.path.abspath("./qs_local"))
    DRIVE = os.path.join(BASE, "drive")

LLAMA   = f"{BASE}/llama.cpp"
BIN     = f"{LLAMA}/build/bin"
WORK    = f"{BASE}/work"                  # big files (GGUFs) live on local disk, not Drive
HF_DIR  = f"{BASE}/phi4mini_hf"
RUN_DIR = f"{DRIVE}/runs_smoke" if SMOKE_TEST else f"{DRIVE}/runs"
for d in (DRIVE, WORK, RUN_DIR):
    os.makedirs(d, exist_ok=True)
N_LIMIT = 10 if SMOKE_TEST else None

# ---- progress: what is finished, what is left ----
def n_done(v, task):
    p = f"{RUN_DIR}/{v}__{task}.jsonl"
    return sum(1 for _ in open(p)) if os.path.exists(p) else 0
TARGET = 10 if SMOKE_TEST else 200
print(f"{'variant':14s} gsm8k  mmlu")
for v in ORDER:
    print(f"{v:14s} {n_done(v,'gsm8k'):4d}/{TARGET} {n_done(v,'mmlu'):4d}/{TARGET}")
if RUN_VARIANTS == "auto":
    RUN_VARIANTS = [v for v in ORDER if min(n_done(v, "gsm8k"), n_done(v, "mmlu")) < TARGET]
print("\nTo run this session:", RUN_VARIANTS if RUN_VARIANTS else "NOTHING — ALL llama.cpp VARIANTS ARE DONE")

# Without Drive: to resume, upload last session's phase2_results.zip to /content (Files panel) before running
if IN_COLAB and not DRIVE_OK and os.path.exists("/content/phase2_results.zip"):
    shutil.unpack_archive("/content/phase2_results.zip", RUN_DIR)
    print("restored previous results from phase2_results.zip")

def sh(cmd, cwd=None):
    # run a shell command, stream its output, stop the notebook if it fails
    print("$", cmd)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-40:]
        if not line.startswith(("[", "  ")) or "error" in line.lower():   # keep build logs short
            print(line, end="")
    if p.wait() != 0:
        print("".join(tail))
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

print("Drive folder:", DRIVE, "| results folder:", RUN_DIR)

In [ ]:
# ==== 3. Get llama.cpp (pinned commit) and build it with CUDA — cached on Drive ====
BIN_CACHE = f"{DRIVE}/llama_bin_{LLAMA_COMMIT[:7]}.tar.gz"

if not os.path.exists(f"{LLAMA}/convert_hf_to_gguf.py"):
    os.makedirs(LLAMA, exist_ok=True)
    sh("git init -q && git remote add origin https://github.com/ggml-org/llama.cpp.git "
       f"&& git fetch -q --depth 1 origin {LLAMA_COMMIT} && git checkout -q FETCH_HEAD", cwd=LLAMA)

if not os.path.exists(f"{BIN}/llama-server") and os.path.exists(BIN_CACHE):
    sh(f"tar -xzf '{BIN_CACHE}' -C '{LLAMA}'")               # restore binaries from an earlier session

if not os.path.exists(f"{BIN}/llama-server"):
    t0 = time.time()
    cuda = "-DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75" if shutil.which("nvcc") or os.path.exists("/usr/local/cuda/bin/nvcc") else ""
    os.environ["PATH"] += ":/usr/local/cuda/bin"
    sh(f"cmake -B build {cuda} -DBUILD_SHARED_LIBS=OFF -DLLAMA_OPENSSL=OFF -DLLAMA_BUILD_TESTS=OFF "
       "-DLLAMA_BUILD_EXAMPLES=OFF -DCMAKE_BUILD_TYPE=Release", cwd=LLAMA)
    sh(f"cmake --build build --config Release -j {os.cpu_count()} --target llama-server llama-quantize", cwd=LLAMA)
    print(f"build took {(time.time()-t0)/60:.1f} min")
    sh(f"tar -czf '{BIN_CACHE}' -C '{LLAMA}' build/bin")    # cache for next session

sh(f"{BIN}/llama-server --version")
devs = subprocess.run([f"{BIN}/llama-server", "--list-devices"], capture_output=True, text=True)
print(devs.stdout + devs.stderr)
if BASE == "/content":
    assert "CUDA0" in devs.stdout + devs.stderr, "llama.cpp was built WITHOUT CUDA — delete the cached tar on Drive and rebuild"

In [ ]:
# ==== 4. Make the GGUF files: HF -> F16 GGUF -> Q8_0 / Q4_K_M ====
# Only what RUN_VARIANTS needs is built. F16 (7.7 GB) stays on local disk; it takes ~5-8 min to regenerate.
CACHE_QUANTS_ON_DRIVE = False    # True = also copy Q8_0/Q4_K_M to Drive (needs ~6.5 GB free Drive space)

def gguf_path(wtype):
    return f"{WORK}/phi4mini-{wtype}.gguf"

def ensure_f16():
    f16 = gguf_path("F16")
    if os.path.exists(f16):
        return f16
    sh(f"{sys.executable} -m pip install -q sentencepiece protobuf")
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL_ID, local_dir=HF_DIR)
    # NB: we deliberately do NOT pip-install llama.cpp's requirements file (it would replace Colab's torch)
    sh(f"{sys.executable} {LLAMA}/convert_hf_to_gguf.py {HF_DIR} --outtype f16 --outfile {f16}")
    return f16

def ensure_gguf(wtype, extra_args=""):
    out = gguf_path(wtype)
    drive_copy = f"{DRIVE}/gguf/{os.path.basename(out)}"
    if os.path.exists(out):
        return out
    if os.path.exists(drive_copy):
        shutil.copy(drive_copy, out); return out
    if wtype == "F16":
        return ensure_f16()
    f16 = ensure_f16()
    if wtype in ("MXFP4", "NVFP4"):
        # all linear layers -> FP4 format; token embedding stays 8-bit (like NF4 kept embeddings in fp16)
        extra_args += " " + " ".join(f"--tensor-type {t}={wtype.lower()}"
                                     for t in ("attn_qkv", "attn_output", "ffn_up", "ffn_down"))
        base_type = "Q8_0"
    else:
        base_type = wtype                  # "Q8_0" or "Q4_K_M"
    sh(f"{BIN}/llama-quantize {extra_args} {f16} {out} {base_type}")
    if CACHE_QUANTS_ON_DRIVE:
        os.makedirs(f"{DRIVE}/gguf", exist_ok=True); shutil.copy(out, drive_copy)
    return out

# only the first file now; the others are built just before they are needed (cell 9)
for v in RUN_VARIANTS[:1]:
    p = ensure_gguf(VARIANTS[v]["weights"])
    print(f"{v:14s} -> {p}  ({os.path.getsize(p)/1024**3:.2f} GB on disk)")
sh(f"df -h {WORK} | tail -1")

In [ ]:
# ==== 5. Benchmark questions — IDENTICAL to Phase 1 (same seed, same 200 + 200) ====
from datasets import load_dataset
random.seed(42)

gsm8k = load_dataset("openai/gsm8k", "main", split="test").shuffle(seed=42).select(range(200))
def extract_gsm8k_answer(answer_field):
    return answer_field.split("####")[-1].strip().replace(",", "")
gsm8k_questions = [{"question": ex["question"], "gold": extract_gsm8k_answer(ex["answer"])} for ex in gsm8k]

mmlu = load_dataset("cais/mmlu", "all", split="test").shuffle(seed=42).select(range(200))
letters = ["A", "B", "C", "D"]
mmlu_questions = []
for ex in mmlu:
    choices_str = "\n".join(f"{letters[i]}. {c}" for i, c in enumerate(ex["choices"]))
    mmlu_questions.append({"question": f"{ex['question']}\n{choices_str}",
                           "gold": letters[ex["answer"]], "subject": ex["subject"]})

# sanity check: must match the first questions printed in the Phase-1 notebook
assert gsm8k_questions[0]["gold"] == "109", "GSM8K subset differs from Phase 1!"
assert mmlu_questions[0]["subject"] == "college_physics", "MMLU subset differs from Phase 1!"
print(len(gsm8k_questions), len(mmlu_questions), "questions — same subsets as Phase 1")

In [ ]:
# ==== 6. Prompts, extractors, server control, GPU-memory sampler ====
import requests

GSM8K_PROMPT = ("Solve this math problem step by step, then give the final numeric answer "
                "on the last line as 'Answer: <number>'.\n\nProblem: {q}\n")
MMLU_PROMPT  = ("Answer this multiple-choice question. Respond with only the letter "
                "(A, B, C, or D) on the last line as 'Answer: <letter>'.\n\nQuestion: {q}\n")

def extract_number(text):
    text = text.replace(",", "")
    m = re.findall(r"Answer:\s*(-?\d+\.?\d*)", text)
    if m: return m[-1]
    nums = re.findall(r"-?\d+\.?\d*", text)
    return nums[-1] if nums else None

def extract_letter(text):
    m = re.findall(r"Answer:\s*([ABCD])", text.upper())
    if m: return m[-1]
    m = re.findall(r"\b([ABCD])\b", text.upper())
    return m[-1] if m else None

# ---- GPU memory: llama.cpp is a separate process, so torch.cuda.max_memory_allocated() can't see it.
# We sample device memory with NVML and subtract the idle baseline measured before the server starts.
try:
    import pynvml
except ImportError:
    sh(f"{sys.executable} -m pip install -q nvidia-ml-py");
    try:
        import pynvml
    except ImportError:
        pynvml = None

class GpuMemSampler:
    def __init__(self, every_s=0.2):
        self.every_s, self.peak, self._stop = every_s, 0, threading.Event()
        self.ok = False
        if pynvml is not None:
            try:
                pynvml.nvmlInit(); self.h = pynvml.nvmlDeviceGetHandleByIndex(0); self.ok = True
            except Exception as e:
                print("NVML not available:", e)
    def used(self):
        return pynvml.nvmlDeviceGetMemoryInfo(self.h).used if self.ok else 0
    def _loop(self):
        while not self._stop.is_set():
            self.peak = max(self.peak, self.used()); time.sleep(self.every_s)
    def start(self):
        self.baseline = self.used(); self.peak = self.baseline
        self._t = threading.Thread(target=self._loop, daemon=True); self._t.start(); return self
    def stop(self):
        self._stop.set(); self._t.join()
        return (self.peak - self.baseline) / 1024**3      # GB used by llama-server

# ---- llama-server lifecycle
BUF_RE = re.compile(r"(\S+)\s+(model|KV|compute|output)\s+buffer size\s*=\s*([\d.]+)\s*MiB")

def start_server(gguf, kv_type, log_path):
    cmd = [f"{BIN}/llama-server", "-m", gguf, "-ngl", "99", "-c", str(CTX_SIZE), "-np", "1",
           "-fa", "on", "-ctk", kv_type, "-ctv", kv_type, "--jinja", "--seed", "42",
           "--host", "127.0.0.1", "--port", str(PORT), "--no-webui", "-lv", "4"]   # -lv 4 = log buffer sizes
    log = open(log_path, "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(600):
        if proc.poll() is not None:
            break
        try:
            if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
                return proc
        except requests.exceptions.RequestException:
            pass
        time.sleep(1)
    print(open(log_path).read()[-3000:])
    proc.kill(); raise RuntimeError("llama-server did not start — see log above")

def stop_server(proc):
    proc.terminate()
    try: proc.wait(timeout=30)
    except subprocess.TimeoutExpired: proc.kill()

def buffer_breakdown(log_path):
    # MiB per buffer kind on the GPU (CUDA0). CUDA_Host = pinned CPU RAM, so it is excluded.
    found = BUF_RE.findall(open(log_path).read())
    on_gpu = any(re.fullmatch(r"CUDA\d+", d) for d, _, _ in found)
    out = {}
    for dev, kind, mib in found:
        if re.fullmatch(r"CUDA\d+", dev) or not on_gpu:
            out[kind] = out.get(kind, 0.0) + float(mib)
    res = {f"{k.lower()}_buf_gb": round(v / 1024, 3) for k, v in out.items()}
    kv_line = re.search(r"llama_kv_cache: size = .*", open(log_path).read())
    res["kv_cache_log"] = kv_line.group(0).strip() if kv_line else None   # proves which KV type was used
    return res

def server_generate(prompt, max_new_tokens):
    payload = {"messages": [{"role": "user", "content": prompt}],
               "max_tokens": max_new_tokens, "temperature": 0.0, "top_k": 1, "seed": 42,
               "cache_prompt": False}          # no prefix reuse between questions (HF runs had none either)
    r = requests.post(f"http://127.0.0.1:{PORT}/v1/chat/completions", json=payload, timeout=1200)
    r.raise_for_status(); j = r.json()
    return j["choices"][0]["message"]["content"], j["usage"]["completion_tokens"], j.get("timings", {})
print("helpers ready")

In [ ]:
# ==== 7. Resumable benchmark loop (same scoring as Phase 1; one JSON line per question on Drive) ====
def run_benchmark_resumable(variant, task, questions, limit=None):
    qs = questions[:limit] if limit else questions
    path = f"{RUN_DIR}/{variant}__{task}.jsonl"
    done = {}
    if os.path.exists(path):
        for line in open(path):
            rec = json.loads(line); done[rec["idx"]] = rec
    todo = [i for i in range(len(qs)) if i not in done]
    print(f"[{variant}|{task}] {len(done)} already done, {len(todo)} to go")
    with open(path, "a") as f:
        for n, i in enumerate(todo):
            ex = qs[i]
            prompt = (GSM8K_PROMPT if task == "gsm8k" else MMLU_PROMPT).format(q=ex["question"])
            t0 = time.time()
            text, n_tok, timings = server_generate(prompt, MAX_NEW_TOKENS)
            dt = time.time() - t0
            pred = extract_number(text) if task == "gsm8k" else extract_letter(text)
            rec = {"idx": i, "gold": ex["gold"], "pred": pred,
                   "correct": (pred is not None) and (str(pred).strip() == str(ex["gold"]).strip()),
                   "n_tokens": n_tok, "wall_s": dt,
                   "decode_tok_s": timings.get("predicted_per_second"), "subject": ex.get("subject"),
                   "prompt_n": timings.get("prompt_n"), "prompt_ms": timings.get("prompt_ms"),
                   "predicted_ms": timings.get("predicted_ms"),
                   "output": text}
            f.write(json.dumps(rec) + "\n"); f.flush(); done[i] = rec
            if (n + 1) % 20 == 0:
                acc = 100 * sum(r["correct"] for r in done.values()) / len(done)
                print(f"  ...{len(done)}/{len(qs)}  running acc {acc:.1f}%")
    return [done[i] for i in sorted(done)]

def summarize(records):
    toks = sum(r["n_tokens"] for r in records); secs = sum(r["wall_s"] for r in records)
    dec = [r["decode_tok_s"] for r in records if r.get("decode_tok_s")]
    return {"accuracy_pct": round(100 * sum(r["correct"] for r in records) / len(records), 2),
            "tokens_per_sec": round(toks / secs, 2),                  # same definition as Phase 1
            "decode_tok_s": round(sum(dec) / len(dec), 2) if dec else None,
            "n_questions": len(records)}

In [ ]:
# ==== 8. Check: llama.cpp applies the SAME chat template as HF apply_chat_template() ====
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
probe = GSM8K_PROMPT.format(q=gsm8k_questions[0]["question"])
hf_text = tok.apply_chat_template([{"role": "user", "content": probe}], tokenize=False, add_generation_prompt=True)

proc = start_server(ensure_gguf("F16"), "f16", f"{WORK}/template_check.log")
try:
    ll_text = requests.post(f"http://127.0.0.1:{PORT}/apply-template",
                            json={"messages": [{"role": "user", "content": probe}]}).json()["prompt"]
finally:
    stop_server(proc)
print("HF      :", repr(hf_text[-120:]))
print("llama.cpp:", repr(ll_text[-120:]))
print("IDENTICAL" if hf_text.strip() == ll_text.strip() else "!! DIFFERENT — tell me before running the benchmark")

In [ ]:
# ==== 9. Run the selected variants ====
import pandas as pd

for v in RUN_VARIANTS:
    cfg = VARIANTS[v]
    print(f"\n==================== {v} ====================")
    gguf = ensure_gguf(cfg["weights"])
    gguf_size_gb = os.path.getsize(gguf) / 1024**3
    log_path = f"{WORK}/{v}.server.log"
    mem = GpuMemSampler().start()
    proc = start_server(gguf, cfg["kv"], log_path)
    try:
        for task, qs in [("gsm8k", gsm8k_questions), ("mmlu", mmlu_questions)]:
            recs = run_benchmark_resumable(v, task, qs, limit=N_LIMIT)
            print(f"[{v}|{task}]", summarize(recs))
    finally:
        stop_server(proc)
        peak_gb = mem.stop()
    meta_path = f"{RUN_DIR}/{v}__meta.json"
    meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {}
    meta.update({"weights": cfg["weights"], "kv": cfg["kv"], "gguf_file_gb": round(os.path.getsize(gguf)/1024**3, 3),
                 "peak_mem_gb": round(max(peak_gb, meta.get("peak_mem_gb", 0)), 3),
                 "llama_commit": LLAMA_COMMIT, **buffer_breakdown(log_path)})
    json.dump(meta, open(meta_path, "w"), indent=1)
    print(v, meta)

In [ ]:
# ==== 10. Collect everything finished so far into one CSV (safe to re-run any time) ====
rows = []
for v in VARIANTS:
    meta_path = f"{RUN_DIR}/{v}__meta.json"
    if not os.path.exists(meta_path):
        continue
    meta = json.load(open(meta_path))
    for task in ("gsm8k", "mmlu"):
        p = f"{RUN_DIR}/{v}__{task}.jsonl"
        if os.path.exists(p):
            recs = [json.loads(l) for l in open(p)]
            rows.append({"variant": v, "task": task, **summarize(recs), "runtime": "llama.cpp", **meta})
df = pd.DataFrame(rows)
df.to_csv(f"{RUN_DIR}/results_phase2_llamacpp.csv", index=False)
cols = ["variant", "task", "accuracy_pct", "tokens_per_sec", "decode_tok_s", "peak_mem_gb",
        "model_buf_gb", "kv_buf_gb", "compute_buf_gb", "n_questions"]
print(df[[c for c in cols if c in df.columns]].to_string(index=False))
print("saved ->", f"{RUN_DIR}/results_phase2_llamacpp.csv")

### After each session
* Send me the table printed by cell 10 (or the CSV from your Drive folder `quant_study_phase2/runs/`).
* If anything fails, send the **last ~30 lines of the error** — the server log is in `/content/work/<variant>.server.log`.
* Not done with all 5 variants? Next session: change `RUN_VARIANTS`, *Run all*. Finished questions are skipped.

In [ ]:
# ==== 12. Time to first token (TTFT) for ALL llama.cpp variants, measured the same way in one session ====
# 30 GSM8K + 30 MMLU prompts per variant, 1 generated token each. TTFT = wall-clock time until the first
# token is returned (prompt processing + first token). llama.cpp's own prompt_ms is recorded as well.
TTFT_PATH = f"{RUN_DIR}/ttft_llamacpp.json"
ttft = json.load(open(TTFT_PATH)) if os.path.exists(TTFT_PATH) else {}
probe_qs = [("gsm8k", GSM8K_PROMPT.format(q=q["question"])) for q in gsm8k_questions[:30]] + \
           [("mmlu", MMLU_PROMPT.format(q=q["question"])) for q in mmlu_questions[:30]]
for v in ORDER:
    if v in ttft:
        continue
    cfg = VARIANTS[v]
    proc = start_server(ensure_gguf(cfg["weights"]), cfg["kv"], f"{WORK}/{v}.ttft.log")
    try:
        server_generate(probe_qs[0][1], 1)                     # warm-up (not counted)
        res = {"gsm8k": [], "mmlu": []}
        for task, prompt in probe_qs:
            t0 = time.time(); _, _, tm = server_generate(prompt, 1); wall = (time.time() - t0) * 1000
            res[task].append({"ttft_ms": wall, "prompt_ms": tm.get("prompt_ms"), "prompt_n": tm.get("prompt_n")})
    finally:
        stop_server(proc)
    import statistics as stx
    ttft[v] = {t: {"median_ttft_ms": round(stx.median(r["ttft_ms"] for r in rs), 1),
                   "median_prompt_ms": round(stx.median(r["prompt_ms"] for r in rs), 1),
                   "median_prompt_tokens": stx.median(r["prompt_n"] for r in rs)} for t, rs in res.items()}
    json.dump(ttft, open(TTFT_PATH, "w"), indent=1)
    print(v, ttft[v])
print("TTFT saved to", TTFT_PATH)

In [ ]:
# ==== 11. Download all results as a zip (ESSENTIAL if you are working without Drive) ====
shutil.make_archive(f"{BASE}/phase2_results", "zip", RUN_DIR)
if IN_COLAB:
    from google.colab import files
    files.download(f"{BASE}/phase2_results.zip")
print("zip:", f"{BASE}/phase2_results.zip")